# 🔍 Notebook 03: Retrieval & Semantic Reranking Pipeline

Trong notebook này, chúng ta sẽ khảo sát và đánh giá tầng truy xuất tài liệu:
1. **Vector Retrieval**: Truy xuất dựa trên độ tương đồng Cosine Similarity trong Qdrant
2. **Semantic Reranking**: Đánh giá lại thứ hạng bằng Cross-Encoder (`BAAI/bge-reranker-v2-m3`)
3. **Metadata Filtering**: Lọc tài liệu theo metadata (`file_name`, `page_number`, `section_title`)
4. **Đo đạc hiệu năng**: Thử nghiệm độ trễ (latency) của tầng Retrieval

## 1. Thiết lập Môi trường & Imports

In [1]:
import sys
import time
from pathlib import Path

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.config import get_config
from src.retrieval.vector_retriever import VectorRetriever
from src.retrieval.reranker import BGEReranker, get_reranker
from src.retrieval.pipeline import RetrievalPipeline

cfg = get_config(reload=True)
print(f"✅ Đã nạp thành công các module Retrieval. Qdrant: {cfg.vector_store.url}")

✅ Đã nạp thành công các module Retrieval. Qdrant: http://localhost:6333


## Format output

In [ ]:
import json
from html import escape
from IPython.display import display, HTML


def display_retrieval_results(results):
    """Hiển thị kết quả retrieval theo từng card trong Jupyter/Kaggle."""

    if not results:
        display(HTML("<p><b>Không tìm thấy kết quả.</b></p>"))
        return

    # Những metadata quan trọng cần xem nhanh
    important_fields = [
        "file_name",
        "file_type",
        "section_title",
        "breadcrumb",
        "section_path",
        "doc_id",
        "chunk_id",
        "chunk_index",
        "token_count",
        "embedding_model",
        "index_collection",
    ]

    for i, item in enumerate(results, start=1):
        metadata = item.metadata or {}
        rank = getattr(item, "rank", i)
        score = float(item.score)
        content = escape(str(item.text or "")).replace("\n", "<br>")

        # Metadata rút gọn
        metadata_rows = ""
        for key in important_fields:
            value = metadata.get(key)

            if value is not None:
                if isinstance(value, (dict, list)):
                    value = json.dumps(value, ensure_ascii=False)

                metadata_rows += f"""
                <tr>
                    <td style="
                        padding:6px 10px;
                        font-weight:600;
                        color:#475569;
                        width:150px;
                        vertical-align:top;
                    ">{escape(key)}</td>
                    <td style="
                        padding:6px 10px;
                        overflow-wrap:anywhere;
                    ">{escape(str(value))}</td>
                </tr>
                """

        # Toàn bộ metadata, bao gồm các trường dài, được thu gọn
        full_metadata = escape(
            json.dumps(metadata, ensure_ascii=False, indent=2, default=str)
        )

        html = f"""
        <div style="
            border:1px solid #cbd5e1;
            border-radius:10px;
            padding:18px;
            margin:14px 0;
            font-family:Arial,sans-serif;
            color:#1e293b;
            background:#ffffff;
        ">
            <div style="
                display:flex;
                justify-content:space-between;
                align-items:center;
                flex-wrap:wrap;
                gap:10px;
                border-bottom:1px solid #e2e8f0;
                padding-bottom:12px;
            ">
                <div style="font-size:17px;font-weight:700;">
                    Kết quả #{rank}
                </div>
                <div style="
                    background:#f1f5f9;
                    padding:7px 12px;
                    border-radius:7px;
                    font-weight:700;
                ">
                    Score: {score:.6f}
                </div>
            </div>

            <h4 style="margin:16px 0 8px;">Nội dung truy xuất</h4>
            <div style="
                line-height:1.7;
                overflow-wrap:anywhere;
                white-space:normal;
            ">{content}</div>

            <h4 style="margin:18px 0 8px;">Metadata chính</h4>
            <table style="
                width:100%;
                border-collapse:collapse;
                font-size:13px;
                line-height:1.5;
            ">
                {metadata_rows}
            </table>

            <details style="margin-top:16px;">
                <summary style="cursor:pointer;font-weight:600;">
                    Xem toàn bộ metadata
                </summary>
                <pre style="
                    margin-top:10px;
                    padding:12px;
                    background:#f8fafc;
                    border:1px solid #e2e8f0;
                    border-radius:6px;
                    white-space:pre-wrap;
                    overflow-wrap:anywhere;
                    font-size:12px;
                    line-height:1.5;
                ">{full_metadata}</pre>
            </details>
        </div>
        """

        display(HTML(html))

## 2. Thử nghiệm Vector Search Đơn thuần (Bi-Encoder)

Vector Retriever nhúng câu hỏi qua `bge-m3` và tìm kiếm các chunks có độ tương đồng Cosine cao nhất.

In [2]:
retriever = VectorRetriever(top_k=10)

2026-10-19 15:56:56 | INFO    | embedder:__init__:39 - Đang tải Embedding model: BAAI/bge-m3 trên thiết bị: cpu


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

2026-10-19 15:57:06 | INFO    | embedder:__init__:49 - Tải model BAAI/bge-m3 thành công. Kích thước vector: 1024


In [ ]:
query = "Accuracy trên tập train của DenseNet201 tại epoch 35 là bao nhiêu?"

print(f"❓ Câu truy vấn: '{query}'\n")
start_time = time.time()
candidate_chunks = retriever.retrieve(query=query, top_k=10)
latency = time.time() - start_time

display_retrieval_results(candidate_chunks)

❓ Câu truy vấn: 'Accuracy trên tập train của DenseNet201 tại epoch 35 là bao nhiêu?'

2026-10-19 15:58:58 | INFO    | vector_retriever:retrieve:65 - Bắt đầu tìm kiếm vector cho câu hỏi: 'Accuracy trên tập train của DenseNet201 tại epoch 35 là bao nhiêu?' (top_k=10)


The `sentences` argument was renamed and is now deprecated. Please use `inputs` instead.


2026-10-19 15:58:59 | INFO    | vector_retriever:retrieve:119 - Đã tìm thấy 10 chunks tương đồng từ Qdrant.
🎯 TÌM THẤY 10 CHUNKS BẰNG VECTOR SEARCH (1.399s):

  • [Hạng #1] - Điểm Cosine: 0.7074 | Nguồn: paper_Phân tích và xử lí ảnh.docx (Kết quả huấn luyện và so sánh)
    Trích đoạn: Mô hình EfficientNetB0 cho thấy tốc độ hội tụ nhanh hơn DenseNet201, chỉ sau 10 epoch mà accuracy trên tập train đã đạt gần 90% nhưng val_loss vẫn còn khá cao. Ở epoch thứ 15 có thể thấy ReduceLROnPlateau đã kích hoạt để giảm tốc độ học xuống còn 1e-5 giúp val_loss bắt đầu ổn định và giảm sâu. Đến epoch 25 có thể thấy accuracy trên tập val đạt gần 90% và val_loss chỉ còn 0.3696 cho thấy mô hình đang được học tốt. Và kết quả cho thấy mô hình có val_loss tốt nhất tại eopch thứ 26....

  • [Hạng #2] - Điểm Cosine: 0.6856 | Nguồn: paper_Phân tích và xử lí ảnh.docx (Biểu đồ loss (train/validation))
    Trích đoạn: Sau epoch 20 có thể thấy giá trị val loss và train loss có xu hướng song song, gần nhau và xoay q

## 3. Thử nghiệm Tái Xếp Hạng với Cross-Encoder Reranker

Mô hình `BAAI/bge-reranker-v2-m3` đánh giá ngữ nghĩa sâu sắc giữa cặp `(query, passage)` cùng một lúc.

In [ ]:
reranker = get_reranker()

In [ ]:

if reranker is not None:
    print(f" Đang tái xếp hạng {len(candidate_chunks)} chunks với {reranker.model_name}...")
    start_time = time.time()
    reranked_chunks = reranker.rerank(query=query, chunks=candidate_chunks, top_n=3)
    rerank_time = time.time() - start_time
    
display_retrieval_results(reranked_chunks)

 Đang tái xếp hạng 10 chunks với BAAI/bge-reranker-v2-m3...
2026-10-14 22:47:51 | INFO    | reranker:rerank:61 - Bắt đầu Reranking cho 10 chunks với câu hỏi: 'Chỉ số Silhouette của mô hình K-Means trên tập Fitness là bao nhiêu?' (top_n=3)
2026-10-14 22:47:56 | INFO    | reranker:rerank:85 - Hoàn tất Reranking. Lọc lại top 3 chunks chất lượng nhất.
🏆 KẾT QUẢ SAU RERANKING (4.719s):

  • [Hạng mới #1] - Điểm Cross-Encoder: 0.9910
    Nguồn: paper_data mining.pdf (F. Kết quả huấn luyện)
    Nội dung:
    # F. Kết quả huấn luyện
Với kết quả k=4 sau khi dùng phương pháp Elbow, kết quả phân cụm với k=4 ở
hai mô hình K-Means và HAC cho thấy hiệu suất của K-Means tốt hơn với điểm
Silhouette Score đạt trên 0.634 trong khi đó HAC chỉ đạt được 0.537 (thấp hơn
gần 0.1). Do đó đối với vấn đề phân cụm khách hàng nhóm chúng em sẽ lựa chọn
mô hình K-Means bộ tham số như sau:...

  • [Hạng mới #2] - Điểm Cross-Encoder: 0.9891
    Nguồn: paper_data mining.pdf (MÔ HÌNH SILHOUETTE SCORE)
    Nội dung:
   

## 4. Thử nghiệm Tìm kiếm kèm Bộ lọc Metadata (Metadata Filtering)

Ví dụ: Chỉ tìm kiếm trong file PDF `paper_data mining.pdf` mà không lấy kết quả từ file Word.

In [ ]:
from qdrant_client.http import models

filter_query = "Hành vi mua sắm của khách hàng"
query_vector = retriever.embedder.embed_query(filter_query)

# Thiết lập bộ lọc Qdrant: chỉ lấy tài liệu có file_name == 'paper_data mining.pdf'
pdf_filter = models.Filter(
    must=[
        models.FieldCondition(
            key="file_name",
            match=models.MatchValue(value="paper_data mining.pdf"),
        )
    ]
)

results = retriever.client.query_points(
    collection_name=retriever.collection_name,
    query=query_vector,
    query_filter=pdf_filter,
    limit=3,
)

print("=" * 75)
print(f"🎯 KẾT QUẢ TÌM KIẾM CHỈ TRONG 'paper_data mining.pdf':\n")
for idx, pt in enumerate(results.points, start=1):
    p = pt.payload
    print(f"  #{idx} [Score: {pt.score:.4f}] Trang {p.get('page_number')} / {p.get('total_pages')}")
    print(f"     Nội dung: {p.get('text')[:150]}...\n")
print("=" * 75)

## 5. Thử nghiệm Đóng gói trong `RetrievalPipeline`

In [ ]:
pipeline = RetrievalPipeline()
results = pipeline.retrieve("Kỹ thuật phân ngưỡng trong phân đoạn ảnh", top_k=8, top_n=3)

print(f"✅ Pipeline đã tự động tìm {len(results)} chunks phù hợp nhất:")
for r in results:
    print(f"  • [Hạng #{r.rank}] Điểm: {r.score:.4f} | Mục: {r.section_title}")